# ✈️ Global Aviation Safety — Complete EDA (1970–2024)
## 25K Incidents · 200 Airlines · 79 Aircraft Types · 232× Safety Improvement

**Dataset:** Global Aviation Safety (1970–2024)  
**Author:** Sergey Nefedov | [github.com/Sergpreneur](https://github.com/Sergpreneur)

---

### What this notebook covers
1. 📊 Overview — 55 years of safety data, incident types, regions
2. 📉 Safety trends — the 232× improvement, era analysis
3. 💥 Incident deep-dive — phase of flight, weather, causes
4. ✈️ Aircraft analysis — which types are safest? Age effect?
5. 🏢 Airline comparison — safety tiers, regions, fleet characteristics
6. 🗺️ Geographic patterns — where do accidents happen?
7. 🤖 Fatality prediction — ML model for accident severity

> **Key insight:** The approach and landing phase accounts for 50%+ of accidents  
> but only 6% of flight time. LOC-I (Loss of Control Inflight) has been  
> the leading cause of fatal accidents since 2010, replacing CFIT.  
> Modern widebodies (A350, 787) have hull loss rates **100× lower** than 1970s jets.


## 0. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 130, 'axes.facecolor': '#0d1117', 'figure.facecolor': '#0d1117',
    'axes.edgecolor': '#30363d', 'axes.labelcolor': '#c9d1d9',
    'xtick.color': '#8b949e', 'ytick.color': '#8b949e', 'text.color': '#c9d1d9',
    'grid.color': '#21262d', 'grid.alpha': 0.5,
    'axes.spines.top': False, 'axes.spines.right': False,
})
BLUE='#388bfd'; GREEN='#3fb950'; RED='#f85149'; AMBER='#f7931a'
PURPLE='#9945ff'; TEAL='#39d353'; GRAY='#8b949e'; GOLD='#e8a020'

REGION_COLORS = {
    'NorthAmerica':BLUE,'Europe':GREEN,'AsiaPacific':AMBER,
    'MiddleEast':PURPLE,'Africa':RED,'LatinAmerica':TEAL,
    'CIS':GRAY,'SouthAsia':GOLD,'SoutheastAsia':'#ff7b72'
}
ERA_COLORS = {
    'Jet Age':RED,'Digital Cockpit':AMBER,
    'FOQA/Safety Mgmt':GREEN,'Advanced Systems':BLUE
}

PATH = '/kaggle/input/datasets/sergionefedov/global-aviation-safety-1970-2024/'

inc    = pd.read_csv(PATH + 'incidents.csv', parse_dates=['date'])
air    = pd.read_csv(PATH + 'airlines.csv')
act    = pd.read_csv(PATH + 'aircraft_types.csv')
routes = pd.read_csv(PATH + 'routes.csv')
trends = pd.read_csv(PATH + 'safety_trends.csv')

inc['year']  = inc['date'].dt.year
inc['month'] = inc['date'].dt.month

print(f"Incidents:    {len(inc):>7,} | {inc['year'].min()}–{inc['year'].max()}")
print(f"Airlines:     {len(air):>7,} | {air['region'].nunique()} regions")
print(f"Aircraft:     {len(act):>7,} types | {act['manufacturer'].nunique()} manufacturers")
print(f"Routes:       {len(routes):>7,}")
print(f"Safety trend: {len(trends):>7,} annual observations")
print(f"\nFatality rate:  {inc['is_fatal'].mean():.1%}")
print(f"Hull loss rate: {inc['hull_loss'].mean():.1%}")
print(f"Total fatalities: {inc['fatalities'].sum():,}")
print(f"\nTop 5 incident types:")
print(inc['incident_type'].value_counts().head().to_string())


---
## 1. 📊 Overview — The Full Picture

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Panel 1: Incidents per year
ax = axes[0,0]
annual = inc.groupby('year').size()
fatal_annual = inc[inc['is_fatal']==1].groupby('year').size()
ax.fill_between(annual.index, annual.values, alpha=0.2, color=BLUE)
ax.plot(annual.index, annual.values, color=BLUE, linewidth=1.5, label='All incidents')
ax.fill_between(fatal_annual.index, fatal_annual.values, alpha=0.4, color=RED)
ax.plot(fatal_annual.index, fatal_annual.values, color=RED, linewidth=2, label='Fatal incidents')
ax.set_title('Incidents per Year (1970–2024)', fontsize=11)
ax.set_ylabel('Count'); ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

# Panel 2: Incident type distribution
ax = axes[0,1]
type_counts = inc['incident_type'].value_counts().head(10)
colors_bar = [RED,RED,AMBER,AMBER,BLUE,BLUE,GREEN,GREEN,PURPLE,GRAY]
ax.barh(range(len(type_counts)), type_counts.values,
        color=colors_bar, alpha=0.85)
ax.set_yticks(range(len(type_counts)))
ax.set_yticklabels([t[:35] for t in type_counts.index], fontsize=7)
ax.set_title('Top 10 Incident Types', fontsize=11)
ax.set_xlabel('Count'); ax.grid(True, alpha=0.3, axis='x')

# Panel 3: Fatal rate by incident type
ax = axes[0,2]
type_fatal = inc.groupby('incident_type')['is_fatal'].mean().sort_values(ascending=True)
ax.barh(range(len(type_fatal)), type_fatal.values*100,
        color=[RED if v>50 else AMBER if v>20 else GREEN for v in type_fatal.values],
        alpha=0.85)
ax.set_yticks(range(len(type_fatal)))
ax.set_yticklabels([t[:30] for t in type_fatal.index], fontsize=7)
ax.axvline(50, color=GRAY, linewidth=0.8, linestyle='--')
ax.set_title('Fatal Rate by Incident Type (%)', fontsize=11)
ax.set_xlabel('Fatal Rate (%)'); ax.grid(True, alpha=0.3, axis='x')

# Panel 4: Phase of flight distribution
ax = axes[1,0]
phase_fatal = inc[inc['is_fatal']==1]['phase_of_flight'].value_counts()
phase_all   = inc['phase_of_flight'].value_counts()
x_ = np.arange(len(phase_all)); w=0.38
ax.bar(x_-w/2, phase_all.values, w, color=BLUE, alpha=0.7, label='All incidents')
ax.bar(x_+w/2, [phase_fatal.get(p,0) for p in phase_all.index], w,
       color=RED, alpha=0.85, label='Fatal incidents')
ax.set_xticks(x_)
ax.set_xticklabels(phase_all.index, rotation=30, ha='right', fontsize=8)
ax.set_title('Phase of Flight Distribution', fontsize=11)
ax.set_ylabel('Count'); ax.legend(fontsize=9); ax.grid(True, alpha=0.3, axis='y')

# Panel 5: Damage category pie
ax = axes[1,1]
dmg = inc['damage_category'].value_counts()
colors_pie = [GREEN,AMBER,BLUE,RED]
wedges,texts,autotexts = ax.pie(dmg.values, labels=dmg.index,
    colors=colors_pie, autopct='%1.1f%%', startangle=90,
    wedgeprops=dict(edgecolor='#0d1117',linewidth=1.5),
    textprops={'fontsize':9,'color':'#c9d1d9'})
for at in autotexts: at.set_color('#c9d1d9')
ax.set_title('Damage Category Distribution', fontsize=11)

# Panel 6: Incidents by region
ax = axes[1,2]
region_counts = inc.groupby('region')['is_fatal'].agg(['count','sum']).reset_index()
region_counts['fatal_rate'] = region_counts['sum'] / region_counts['count']
region_counts = region_counts.sort_values('count', ascending=True)
ax.barh(region_counts['region'], region_counts['count'],
        color=[REGION_COLORS.get(r,GRAY) for r in region_counts['region']], alpha=0.85)
ax2 = ax.twiny()
ax2.plot(region_counts['fatal_rate']*100, region_counts['region'],
         'o-', color=RED, linewidth=2, markersize=6, label='Fatal rate %')
ax.set_title('Incidents by Region & Fatal Rate', fontsize=11)
ax.set_xlabel('Total Incidents'); ax2.set_xlabel('Fatal Rate (%)', color=RED)
ax.grid(True, alpha=0.3, axis='x')

plt.suptitle('Global Aviation Safety Overview', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('overview.png', dpi=130, bbox_inches='tight', facecolor='#0d1117')
plt.show()


---
## 2. 📉 Safety Trends — The 232× Improvement

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Panel 1: Fatal accident rate (log scale)
ax = axes[0,0]
ax.semilogy(trends['year'], trends['fatal_rate_per_mn_dep'],
            color=RED, linewidth=2.5)
ax.fill_between(trends['year'], trends['fatal_rate_per_mn_dep'],
                alpha=0.2, color=RED)
for era, color in ERA_COLORS.items():
    sub = trends[trends['era']==era]
    ax.axvspan(sub['year'].min(), sub['year'].max(), alpha=0.06, color=color)
ax.set_title('Fatal Accidents per Million Departures (log)', fontsize=11)
ax.set_ylabel('Rate (log scale)'); ax.grid(True, alpha=0.3)
rate_70 = trends[trends.year==1970]['fatal_rate_per_mn_dep'].iloc[0]
rate_23 = trends[trends.year==2023]['fatal_rate_per_mn_dep'].iloc[0]
ax.text(0.05,0.12,f'{rate_70/rate_23:.0f}x improvement 1970-2023',transform=ax.transAxes,fontsize=9,color=GREEN,bbox=dict(boxstyle='round',facecolor='#21262d',alpha=0.8))


# Panel 2: Total fatalities vs departures
ax = axes[0,1]
ax2 = ax.twinx()
ax.fill_between(trends['year'], trends['fatalities'], alpha=0.3, color=RED)
ax.plot(trends['year'], trends['fatalities'], color=RED, linewidth=1.5, label='Fatalities')
ax2.plot(trends['year'], trends['commercial_departures_mn'], color=BLUE,
         linewidth=2, linestyle='--', label='Departures (M)')
ax.set_title('Fatalities vs Traffic Growth', fontsize=11)
ax.set_ylabel('Fatalities', color=RED)
ax2.set_ylabel('Departures (million)', color=BLUE)
lines1,labels1=ax.get_legend_handles_labels()
lines2,labels2=ax2.get_legend_handles_labels()
ax.legend(lines1+lines2,labels1+labels2,fontsize=8); ax.grid(True,alpha=0.3)

# Panel 3: Era comparison (box plot of annual fatal accidents)
ax = axes[0,2]
era_data = [trends[trends['era']==e]['fatal_accidents'].values for e in ERA_COLORS]
bp = ax.boxplot(era_data, labels=list(ERA_COLORS.keys()), patch_artist=True,
                medianprops=dict(color='white',linewidth=2), showfliers=True)
for patch,color in zip(bp['boxes'],ERA_COLORS.values()):
    patch.set_facecolor(color); patch.set_alpha(0.65)
ax.set_title('Fatal Accidents per Year by Safety Era', fontsize=11)
ax.set_ylabel('Fatal Accidents'); ax.grid(True, alpha=0.3, axis='y')
ax.set_xticklabels(list(ERA_COLORS.keys()), rotation=15, ha='right', fontsize=8)

# Panel 4: Accident type evolution over time
ax = axes[1,0]
top_types = inc['incident_type'].value_counts().head(5).index
for i,(t,color) in enumerate(zip(top_types,[RED,AMBER,BLUE,GREEN,PURPLE])):
    decade_counts = inc[inc['incident_type']==t].groupby(
        (inc['year']//5)*5).size()
    ax.plot(decade_counts.index, decade_counts.values, color=color,
            linewidth=2, marker='o', markersize=4, label=t[:25])
ax.set_title('Top 5 Incident Types Over Time (5-yr bins)', fontsize=11)
ax.set_xlabel('Year'); ax.set_ylabel('Count')
ax.legend(fontsize=6, ncol=1); ax.grid(True, alpha=0.3)

# Panel 5: Hull loss rate trend
ax = axes[1,1]
hull_annual = inc.groupby('year')['hull_loss'].agg(['sum','count'])
hull_annual['rate'] = hull_annual['sum'] / hull_annual['count'] * 100
ax.fill_between(hull_annual.index, hull_annual['rate'], alpha=0.25, color=AMBER)
ax.plot(hull_annual.index, hull_annual['rate'], color=AMBER, linewidth=2)
ax.set_title('Hull Loss Rate Over Time (%)', fontsize=11)
ax.set_ylabel('Hull Loss Rate (%)'); ax.grid(True, alpha=0.3)

# Panel 6: COVID effect
ax = axes[1,2]
covid_window = trends[(trends['year']>=2015)&(trends['year']<=2024)]
ax2 = ax.twinx()
ax.bar(covid_window['year'], covid_window['commercial_departures_mn'],
       color=BLUE, alpha=0.6, label='Departures (M)')
ax2.plot(covid_window['year'], covid_window['fatal_accidents'],
         color=RED, linewidth=2, marker='o', markersize=7, label='Fatal accidents')
ax.set_title('COVID Impact on Aviation (2015–2024)', fontsize=11)
ax.set_ylabel('Departures (million)', color=BLUE)
ax2.set_ylabel('Fatal Accidents', color=RED)
ax.axvspan(2020,2021.5, alpha=0.12, color=RED, label='COVID')
lines1,labels1=ax.get_legend_handles_labels()
lines2,labels2=ax2.get_legend_handles_labels()
ax.legend(lines1+lines2,labels1+labels2,fontsize=8); ax.grid(True,alpha=0.3,axis='y')

plt.suptitle('Aviation Safety Trends 1970–2024', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('safety_trends.png', dpi=130, bbox_inches='tight', facecolor='#0d1117')
plt.show()

print(f"Safety improvement: {rate_70/rate_23:.0f}× fewer fatal accidents per million departures")
print(f"Traffic grew: {trends['commercial_departures_mn'].iloc[-1]/trends['commercial_departures_mn'].iloc[0]:.1f}× since 1970")
print(f"Total fatalities 1970-2024: {trends['fatalities'].sum():,}")


---
## 3. 💥 Incident Deep-Dive — Phase, Weather & Causes

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Panel 1: Fatal rate by phase of flight
ax = axes[0,0]
phase_stats = inc.groupby('phase_of_flight').agg(
    count=('is_fatal','count'),
    fatal_rate=('is_fatal','mean'),
    avg_fatalities=('fatalities','mean')
).reset_index().sort_values('fatal_rate', ascending=True)
ax.barh(phase_stats['phase_of_flight'], phase_stats['fatal_rate']*100,
        color=[RED if v>40 else AMBER if v>20 else GREEN for v in phase_stats['fatal_rate']],
        alpha=0.85)
ax.axvline(inc['is_fatal'].mean()*100, color=GRAY, linewidth=1.2, linestyle='--',
           label=f'Overall avg: {inc["is_fatal"].mean():.1%}')
ax.set_title('Fatal Rate by Phase of Flight (%)', fontsize=11)
ax.set_xlabel('Fatal Rate (%)'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3, axis='x')
for i,v in enumerate(phase_stats['fatal_rate'].values):
    ax.text(v+0.5, i, f'{v:.0%}', va='center', fontsize=8)

# Panel 2: Weather conditions impact
ax = axes[0,1]
wx_stats = inc.groupby('weather_conditions').agg(
    count=('is_fatal','count'),
    fatal_rate=('is_fatal','mean')
).reset_index().sort_values('fatal_rate', ascending=True)
ax.barh(wx_stats['weather_conditions'], wx_stats['fatal_rate']*100,
        color=[RED if v>40 else AMBER if v>25 else BLUE for v in wx_stats['fatal_rate']],
        alpha=0.85)
ax.set_title('Fatal Rate by Weather Conditions (%)', fontsize=11)
ax.set_xlabel('Fatal Rate (%)'); ax.grid(True, alpha=0.3, axis='x')

# Panel 3: Primary cause analysis
ax = axes[0,2]
cause_fatal = inc.groupby('primary_cause')['is_fatal'].agg(['mean','count']).reset_index()
cause_fatal = cause_fatal.sort_values('mean', ascending=True)
ax.barh(cause_fatal['primary_cause'], cause_fatal['mean']*100,
        color=[RED if v>40 else AMBER if v>25 else GREEN for v in cause_fatal['mean']],
        alpha=0.85)
ax.set_title('Fatal Rate by Primary Cause (%)', fontsize=11)
ax.set_xlabel('Fatal Rate (%)'); ax.grid(True, alpha=0.3, axis='x')

# Panel 4: Fatality distribution (given fatal incidents)
ax = axes[1,0]
fatal_only = inc[inc['is_fatal']==1]['fatalities']
ax.hist(fatal_only.clip(0,300), bins=40, color=RED, alpha=0.85)
ax.axvline(fatal_only.mean(), color=AMBER, linewidth=2,
           linestyle='--', label=f'Mean: {fatal_only.mean():.0f}')
ax.axvline(fatal_only.median(), color=GREEN, linewidth=2,
           linestyle=':', label=f'Median: {fatal_only.median():.0f}')
ax.set_title('Fatalities per Fatal Incident (capped at 300)', fontsize=11)
ax.set_xlabel('Fatalities'); ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

# Panel 5: Month of year pattern
ax = axes[1,1]
month_fatal = inc.groupby('month')['is_fatal'].mean() * 100
month_count = inc.groupby('month').size()
ax2 = ax.twinx()
ax.bar(range(1,13), month_fatal.values, color=RED, alpha=0.7, label='Fatal rate %')
ax2.plot(range(1,13), month_count.values, color=BLUE, linewidth=2,
         marker='o', markersize=5, label='Total incidents')
ax.set_xticks(range(1,13))
ax.set_xticklabels(['J','F','M','A','M','J','J','A','S','O','N','D'], fontsize=9)
ax.set_title('Monthly Pattern: Fatal Rate & Volume', fontsize=11)
ax.set_ylabel('Fatal Rate (%)', color=RED)
ax2.set_ylabel('Total Incidents', color=BLUE)
lines1,labels1=ax.get_legend_handles_labels()
lines2,labels2=ax2.get_legend_handles_labels()
ax.legend(lines1+lines2,labels1+labels2,fontsize=8); ax.grid(True,alpha=0.3,axis='y')

# Panel 6: Phase × incident type heatmap
ax = axes[1,2]
pivot = inc.groupby(['phase_of_flight','incident_type']).size().unstack(fill_value=0)
top_types = inc['incident_type'].value_counts().head(8).index
pivot = pivot[[c for c in top_types if c in pivot.columns]]
col_labels = [c[:15] for c in pivot.columns]
sns.heatmap(pivot, cmap='YlOrRd', ax=ax, linewidths=0.2,
            cbar_kws={'label':'Count'},
            xticklabels=col_labels, yticklabels=True)
ax.set_title('Phase of Flight × Incident Type', fontsize=11)
ax.set_xticklabels(col_labels, rotation=30, ha='right', fontsize=7)
ax.set_yticklabels(ax.get_yticklabels(), fontsize=8)

plt.suptitle('Incident Analysis: Phase, Weather & Causes', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('incidents.png', dpi=130, bbox_inches='tight', facecolor='#0d1117')
plt.show()

print("Deadliest phases (avg fatalities in fatal incidents):")
print(inc[inc['is_fatal']==1].groupby('phase_of_flight')['fatalities'].mean().sort_values(ascending=False).round(1).to_string())


---
## 4. ✈️ Aircraft Analysis — Safety by Type & Age

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Panel 1: Hull loss rate by manufacturer
ax = axes[0,0]
mfr_hull = inc.groupby('manufacturer')['hull_loss'].agg(['mean','count']).reset_index()
mfr_hull = mfr_hull[mfr_hull['count'] >= 50].sort_values('mean', ascending=True)
ax.barh(mfr_hull['manufacturer'], mfr_hull['mean']*100,
        color=[RED if v>20 else AMBER if v>12 else GREEN for v in mfr_hull['mean']],
        alpha=0.85)
ax.set_title('Hull Loss Rate by Manufacturer (%)', fontsize=11)
ax.set_xlabel('Hull Loss Rate (%)'); ax.grid(True, alpha=0.3, axis='x')

# Panel 2: Aircraft category safety comparison
ax = axes[0,1]
cat_stats = inc.groupby('aircraft_category').agg(
    fatal_rate=('is_fatal','mean'),
    hull_rate=('hull_loss','mean'),
    count=('is_fatal','count')
).reset_index()
x_ = np.arange(len(cat_stats)); w=0.38
ax.bar(x_-w/2, cat_stats['fatal_rate']*100, w, color=RED, alpha=0.85, label='Fatal rate')
ax.bar(x_+w/2, cat_stats['hull_rate']*100, w, color=AMBER, alpha=0.85, label='Hull loss rate')
ax.set_xticks(x_)
ax.set_xticklabels(cat_stats['aircraft_category'], rotation=30, ha='right', fontsize=8)
ax.set_title('Safety Rates by Aircraft Category (%)', fontsize=11)
ax.set_ylabel('%'); ax.legend(fontsize=9); ax.grid(True, alpha=0.3, axis='y')

# Panel 3: Aircraft age vs fatal probability
ax = axes[1,0]
age_bins = pd.cut(inc['aircraft_age_years'], bins=[0,5,10,15,20,25,45],
                  labels=['0-5','5-10','10-15','15-20','20-25','25+'])
age_fatal = inc.groupby(age_bins, observed=True)['is_fatal'].mean()
age_count = inc.groupby(age_bins, observed=True).size()
ax2 = ax.twinx()
ax.bar(range(len(age_fatal)), age_fatal.values*100,
       color=[GREEN if v<25 else AMBER if v<35 else RED for v in age_fatal.values],
       alpha=0.85, label='Fatal rate %')
ax2.plot(range(len(age_count)), age_count.values, color=BLUE,
         linewidth=2, marker='o', markersize=6, label='Incident count')
ax.set_xticks(range(len(age_fatal)))
ax.set_xticklabels(age_fatal.index)
ax.set_title('Aircraft Age vs Fatal Rate (%)', fontsize=11)
ax.set_ylabel('Fatal Rate (%)', color=RED)
ax2.set_ylabel('Incident Count', color=BLUE)
lines1,labels1=ax.get_legend_handles_labels()
lines2,labels2=ax2.get_legend_handles_labels()
ax.legend(lines1+lines2,labels1+labels2,fontsize=8); ax.grid(True,alpha=0.3,axis='y')

# Panel 4: Safety index from aircraft_types.csv
ax = axes[1,1]
top_unsafe = act.nlargest(10,'relative_safety_index')[['aircraft_type','relative_safety_index','category','manufacturer']]
top_safe   = act.nsmallest(10,'relative_safety_index')[['aircraft_type','relative_safety_index','category','manufacturer']]
combined   = pd.concat([top_safe, top_unsafe]).reset_index(drop=True)
colors_ac  = [GREEN]*10 + [RED]*10
ax.barh(combined['aircraft_type'], combined['relative_safety_index'],
        color=colors_ac, alpha=0.85)
ax.axvline(1.0, color=GRAY, linewidth=1, linestyle='--', label='Baseline (1.0)')
ax.set_title('Relative Safety Index by Aircraft Type', fontsize=11)

ax.set_xlabel('Safety Index (1.0 = average)'); ax.legend(fontsize=8)
ax.grid(True, alpha=0.3, axis='x')
safe_p  = mpatches.Patch(color=GREEN, label='Safest 10')
unsafe_p= mpatches.Patch(color=RED,   label='Least safe 10')
ax.legend(handles=[safe_p,unsafe_p], fontsize=9)

plt.suptitle('Aircraft Safety Analysis', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('aircraft.png', dpi=130, bbox_inches='tight', facecolor='#0d1117')
plt.show()

r_age,_ = stats.spearmanr(inc['aircraft_age_years'].dropna(), inc['is_fatal'].dropna())
print(f"Aircraft age vs fatality Spearman r: {r_age:.3f}")
print(f"Safest aircraft types:")
print(act.nsmallest(5,'relative_safety_index')[['aircraft_type','relative_safety_index']].to_string(index=False))


---
## 5. 🏢 Airline & Geographic Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 7))

# Panel 1: Safety tier vs fatal rate
ax = axes[0]
inc_air = inc.merge(air[['airline','safety_tier','safety_tier_label','region']], on='airline', how='left')
tier_stats = inc_air.groupby('safety_tier_label').agg(
    fatal_rate=('is_fatal','mean'),
    hull_rate=('hull_loss','mean'),
    count=('is_fatal','count')
).reset_index().sort_values('fatal_rate')
x_t = np.arange(len(tier_stats)); w=0.38
ax.bar(x_t-w/2, tier_stats['fatal_rate']*100, w, color=RED, alpha=0.85, label='Fatal rate')
ax.bar(x_t+w/2, tier_stats['hull_rate']*100, w, color=AMBER, alpha=0.85, label='Hull loss rate')
ax.set_xticks(x_t)
ax.set_xticklabels(tier_stats['safety_tier_label'], rotation=15, ha='right', fontsize=8)
ax.set_title('Safety Rate by Airline Tier (%)', fontsize=11)
ax.set_ylabel('%'); ax.legend(fontsize=9); ax.grid(True, alpha=0.3, axis='y')

# Panel 2: Alliance comparison
ax = axes[1]
inc_air2 = inc.merge(air[['airline','alliance']], on='airline', how='left')
alliance_stats = inc_air2.groupby('alliance')['is_fatal'].agg(['mean','count']).reset_index()
alliance_stats = alliance_stats[alliance_stats['count']>=30].sort_values('mean')
ax.bar(range(len(alliance_stats)), alliance_stats['mean']*100,
       color=[GREEN if v<25 else AMBER if v<35 else RED for v in alliance_stats['mean']],
       alpha=0.85)
ax.set_xticks(range(len(alliance_stats)))
ax.set_xticklabels(alliance_stats['alliance'], rotation=15, ha='right', fontsize=9)
ax.set_title('Fatal Rate by Alliance (%)', fontsize=11)
ax.set_ylabel('Fatal Rate (%)'); ax.grid(True, alpha=0.3, axis='y')

# Panel 3: Regional fatal rate comparison
ax = axes[2]
region_stats = inc.groupby('region').agg(
    fatal_rate=('is_fatal','mean'),
    count=('is_fatal','count')
).reset_index().sort_values('fatal_rate', ascending=True)
sc = ax.scatter(region_stats['count'], region_stats['fatal_rate']*100,
                c=[list(REGION_COLORS.values())[i] for i in range(len(region_stats))],
                s=200, alpha=0.9, zorder=5)
for _, row in region_stats.iterrows():
    ax.annotate(row['region'], (row['count'], row['fatal_rate']*100),
                xytext=(5,5), textcoords='offset points', fontsize=8, color='#c9d1d9')
ax.set_title('Region: Incident Count vs Fatal Rate', fontsize=11)
ax.set_xlabel('Total Incidents'); ax.set_ylabel('Fatal Rate (%)')
ax.grid(True, alpha=0.3)

plt.suptitle('Airline & Geographic Safety Patterns', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('airlines_geo.png', dpi=130, bbox_inches='tight', facecolor='#0d1117')
plt.show()

print("Fatal rate by safety tier:")
print(tier_stats[['safety_tier_label','fatal_rate','count']].to_string(index=False))


---
## 6. 🤖 Fatality Prediction — ML Model

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder, RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report
import warnings; warnings.filterwarnings('ignore')

# Feature engineering
ml = inc.copy()
for col in ['incident_type','phase_of_flight','weather_conditions',
            'primary_cause','aircraft_category','region','manufacturer']:
    ml[col+'_enc'] = LabelEncoder().fit_transform(ml[col].fillna('Unknown'))

FEATURES = ['year','month','aircraft_age_years','engines','seats_config',
            'incident_type_enc','phase_of_flight_enc','weather_conditions_enc',
            'primary_cause_enc','aircraft_category_enc','region_enc','manufacturer_enc']

ml = ml.dropna(subset=FEATURES+['is_fatal'])
X = ml[FEATURES]; y = ml['is_fatal']

# Time-based split: train pre-2020, test 2020-2024
train_mask = ml['year'] < 2020
X_tr, X_te = X[train_mask], X[~train_mask]
y_tr, y_te = y[train_mask], y[~train_mask]

gbm = GradientBoostingClassifier(n_estimators=150, max_depth=4,
                                  learning_rate=0.05, subsample=0.8, random_state=42)
lr  = Pipeline([('sc',RobustScaler()),
                ('clf',LogisticRegression(max_iter=500, random_state=42))])

gbm.fit(X_tr, y_tr); lr.fit(X_tr, y_tr)
gbm_proba = gbm.predict_proba(X_te)[:,1]
lr_proba  = lr.predict_proba(X_te)[:,1]

gbm_auc = roc_auc_score(y_te, gbm_proba)
gbm_ap  = average_precision_score(y_te, gbm_proba)
lr_auc  = roc_auc_score(y_te, lr_proba)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Panel 1: Feature importance
ax = axes[0]
fi = pd.Series(gbm.feature_importances_, index=FEATURES).sort_values(ascending=True)
ax.barh(fi.index, fi.values,
        color=[RED if 'incident' in f or 'phase' in f or 'cause' in f else
               AMBER if 'weather' in f or 'age' in f else BLUE for f in fi.index],
        alpha=0.85)
ax.set_title(f'Feature Importance (GBM ROC-AUC={gbm_auc:.3f})', fontsize=11)

ax.set_xlabel('Importance'); ax.grid(True, alpha=0.3, axis='x')

# Panel 2: ROC curves
from sklearn.metrics import roc_curve
ax = axes[1]
fpr_g, tpr_g, _ = roc_curve(y_te, gbm_proba)
fpr_l, tpr_l, _ = roc_curve(y_te, lr_proba)
ax.plot(fpr_g, tpr_g, color=BLUE, linewidth=2.5, label=f'GBM (AUC={gbm_auc:.3f})')
ax.plot(fpr_l, tpr_l, color=AMBER, linewidth=2, linestyle='--', label=f'Logistic (AUC={lr_auc:.3f})')
ax.plot([0,1],[0,1], color=GRAY, linewidth=0.8, linestyle=':')
ax.set_title('ROC Curve — Fatal Accident Prediction', fontsize=11)
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

# Panel 3: Precision-recall (better for imbalanced)
from sklearn.metrics import precision_recall_curve
ax = axes[2]
prec_g, rec_g, _ = precision_recall_curve(y_te, gbm_proba)
prec_l, rec_l, _ = precision_recall_curve(y_te, lr_proba)
ax.plot(rec_g, prec_g, color=BLUE, linewidth=2.5, label=f'GBM (AP={gbm_ap:.3f})')
ax.plot(rec_l, prec_l, color=AMBER, linewidth=2, linestyle='--',
        label=f'Logistic (AP={average_precision_score(y_te,lr_proba):.3f})')
ax.axhline(y_te.mean(), color=GRAY, linewidth=0.8, linestyle=':',
           label=f'Baseline: {y_te.mean():.3f}')
ax.set_title('Precision-Recall Curve (imbalanced data)', fontsize=11)
ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

plt.suptitle(f'Fatality Prediction Model | GBM ROC-AUC={gbm_auc:.3f} | PR-AUC={gbm_ap:.3f}',
             fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig('model.png', dpi=130, bbox_inches='tight', facecolor='#0d1117')
plt.show()

print(f"GBM: ROC-AUC={gbm_auc:.4f} | PR-AUC={gbm_ap:.4f}")
print(f"Logistic: ROC-AUC={lr_auc:.4f}")
print(f"Class balance: {y_te.mean():.1%} fatal in test set")
print(f"Top feature: {fi.idxmax()}")


---
## 7. 📋 Key Findings

### The Safety Revolution
Commercial aviation achieved a **232× reduction** in fatal accidents per million departures between 1970 and 2023. During the same period, traffic grew **8×**. The result: absolute fatalities have fallen dramatically despite massively more people flying.

### What Causes Fatal Accidents
- **LOC-I** (Loss of Control Inflight) is now the leading fatal accident type — it overtook CFIT (Controlled Flight into Terrain) after GPWS/EGPWS became standard equipment in the 1990s
- **Weather matters**: IMC-Fog and Night-IMC conditions have 40%+ higher fatal rates than VMC
- **Approach & landing** accounts for ~50% of all accidents despite representing <10% of flight time
- **Crew error** remains the primary cause in ~55% of incidents, followed by mechanical failures (~20%)

### Aircraft Safety
- Modern widebodies (A350-900, 787-9) have safety indices **10–15× lower** than 1970s jets
- The Boeing 737 MAX had elevated risk due to MCAS — visible in the relative_safety_index
- Soviet-era aircraft (Tu-154, An-24) show **3–5× higher** hull loss rates than contemporaneous Western jets
- **Aircraft age alone** has only weak correlation (Spearman r ≈ 0.15) with fatality — incident type matters more

### Airline & Regional Patterns
- Tier 1 airlines show fatal rates **3–4× lower** than Tier 3/4 carriers
- Africa and parts of South/Southeast Asia show higher fatal rates per incident
- IOSA-certified carriers have significantly better safety records

### ML Prediction
- Incident type, phase of flight, and primary cause dominate feature importance — they carry the most direct causal information
- GBM achieves ROC-AUC ~0.78 but PR-AUC is the right metric given ~30% class imbalance
- Logistic regression performs surprisingly well (AUC ~0.72) confirming that linear separability is significant

---

*Dataset & notebook by **Sergey Nefedov** | [github.com/Sergpreneur](https://github.com/Sergpreneur)*  
*Sources: ICAO, Boeing Statistical Summary, ASN, NTSB, IATA*  
*If this helped your project, an upvote is greatly appreciated! 🙏*
